# 04 - Extração estruturada via LLM

Para cada PDF baixado no notebook 03, extrai full text com `pymupdf` e roda extração estruturada via Claude Sonnet 4.6 produzindo as linhas da tabela `configurations` (schema em `00_design.ipynb`).

**Saída:**
- `data/processed/extraction_checkpoints.jsonl` - uma linha por paper com a resposta crua do LLM.
- `data/processed/configurations_raw.parquet` - base achatada: uma linha por configuração experimental extraída (1 paper → N linhas), com todos os campos `*_raw`.

## Decisões

- **Parser de PDF:** `pymupdf` (rápido, simples). Se descobrirmos que tabelas de resultados estão sendo perdidas, escalamos para GROBID nos casos problemáticos.
- **Truncagem inteligente:** muitos papers passam de 50K chars. Para conter custo, truncamos para ~30K chars total = primeiros 10K (intro + métodos) + últimos 20K (experimentos + resultados + conclusão). Também tentamos cortar a seção de Referências quando detectada.
- **Modelo LLM:** `claude-sonnet-4-6` ($3/M input + $15/M output) - extração estruturada com muitos campos por linha; erro propaga até a base final, vale o custo extra vs Haiku.
- **Structured output:** `client.messages.parse()` com schema Pydantic aninhado (`PaperExtraction` contendo `list[Configuration]`).
- **Rate limit:** mesmo limiter do notebook 02 (45 RPM, margem abaixo dos 50).

In [ ]:
!pip install --upgrade pymupdf

In [ ]:
import os
import re
import json
import time
import threading
from collections import deque
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List, Literal

import pandas as pd
import pymupdf
from tqdm.auto import tqdm
from pydantic import BaseModel, Field
import anthropic

assert os.environ.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

In [ ]:
NB_DIR = Path.cwd()
PROJECT_DIR = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PDF_DIR = PROJECT_DIR / "pdfs"

MODEL = "claude-sonnet-4-6"
N_WORKERS = 4
RPM_LIMIT = 45
MAX_RETRIES = 5
MAX_TOKENS_OUTPUT = 8000

# Truncagem do texto enviado
MAX_TEXT_CHARS = 30000
HEAD_CHARS = 10000  # do começo (intro, métodos)
# TAIL_CHARS = MAX_TEXT_CHARS - HEAD_CHARS (calculado em runtime)

INPUT_PATH = PROCESSED_DIR / "papers_with_pdf.parquet"
CKPT_PATH = PROCESSED_DIR / "extraction_checkpoints.jsonl"
CONFIG_PATH = PROCESSED_DIR / "configurations_raw.parquet"

print(f"Modelo: {MODEL}")
print(f"Rate limit: {RPM_LIMIT} req/min, {N_WORKERS} workers")
print(f"Truncagem: head={HEAD_CHARS} + tail={MAX_TEXT_CHARS - HEAD_CHARS} = {MAX_TEXT_CHARS} chars")
print(f"\nInput:  {INPUT_PATH}")
print(f"Ckpt:   {CKPT_PATH}")
print(f"Saída:  {CONFIG_PATH}")

In [ ]:
class RateLimiter:
    """Sliding-window rate limiter (mesmo do notebook 02)."""

    def __init__(self, max_per_minute: int):
        self.max_per_minute = max_per_minute
        self.timestamps: deque = deque()
        self.lock = threading.Lock()

    def acquire(self):
        while True:
            with self.lock:
                now = time.monotonic()
                while self.timestamps and self.timestamps[0] <= now - 60:
                    self.timestamps.popleft()
                if len(self.timestamps) < self.max_per_minute:
                    self.timestamps.append(now)
                    return
                sleep_time = self.timestamps[0] + 60 - now + 0.05
            time.sleep(max(sleep_time, 0.01))


rate_limiter = RateLimiter(RPM_LIMIT)
print(f"RateLimiter ativo: {RPM_LIMIT} req/min")

In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(f"Total: {len(df)}")
print(f"\nStatus do PDF:")
print(df["pdf_status"].value_counts(dropna=False))

df_ext = df[df["pdf_status"].isin(["success", "already_exists"])].copy().reset_index(drop=True)
print(f"\nA extrair: {len(df_ext)}")

## Extração de texto do PDF

`pymupdf` é rápido (~30ms/paper). Estratégia de truncagem:

1. Extrai texto de todas as páginas.
2. Procura cabeçalho de Referências (`\nReferences`, `\nREFERENCES`, `\nBibliography`) na segunda metade do texto — se encontrar, descarta da posição em diante.
3. Se o texto restante for ≤ MAX_TEXT_CHARS, devolve inteiro.
4. Caso contrário, devolve `[head] + "[...middle truncated...]" + [tail]` — mantém início (abstract, intro, métodos) e fim (resultados, discussão, conclusão).

In [ ]:
REFERENCES_PATTERNS = ["\nReferences\n", "\nREFERENCES\n", "\nReferences \n",
                       "\nBibliography\n", "\nBIBLIOGRAPHY\n", "\nReferences:\n"]


def extract_pdf_text(pdf_path: str) -> str:
    """Extrai todo o texto do PDF concatenado por página."""
    doc = pymupdf.open(pdf_path)
    try:
        chunks = []
        for page in doc:
            chunks.append(page.get_text())
        return "\n".join(chunks)
    finally:
        doc.close()


def smart_truncate(text: str, max_chars: int = MAX_TEXT_CHARS,
                    head_chars: int = HEAD_CHARS) -> str:
    """Strip references, then head+tail truncate se ainda for grande."""
    # Strip references (procura na segunda metade)
    half = len(text) // 2
    for pat in REFERENCES_PATTERNS:
        idx = text.rfind(pat)
        if idx > half:
            text = text[:idx]
            break

    if len(text) <= max_chars:
        return text
    tail_chars = max_chars - head_chars
    return (text[:head_chars]
            + "\n\n[...middle truncated for length...]\n\n"
            + text[-tail_chars:])


# Sanity check — extrai e mostra estatísticas do primeiro paper
if len(df_ext) > 0:
    sample_path = df_ext.iloc[0]["pdf_local_path"]
    print(f"Teste com: {sample_path}")
    raw = extract_pdf_text(sample_path)
    truncated = smart_truncate(raw)
    print(f"  raw:       {len(raw):>7} chars (~{len(raw)//4:>5} tokens)")
    print(f"  truncated: {len(truncated):>7} chars (~{len(truncated)//4:>5} tokens)")
    print(f"\nPrimeiros 500 chars:\n{truncated[:500]}")

## Schema da extração

Cada paper pode produzir múltiplas `Configuration`. O LLM devolve um `PaperExtraction` com lista de configs + um `paper_notes` opcional (útil quando o paper acaba não tendo dado experimental utilizável).

Todos os campos `*_raw` são strings literais do paper — normalização é separada (notebook 05). Os 3 campos meta (`extracted_evidence`, `extraction_confidence`, `notes`) são auxiliares pra auditoria.

In [ ]:
class Configuration(BaseModel):
    # Identificação da configuração (obrigatórios)
    dataset_name_raw: str = Field(description="Dataset name verbatim from the paper.")
    model_name_raw: str = Field(description="Model architecture name verbatim from the paper.")
    balancing_strategy_raw: str = Field(description="Class-balancing strategy verbatim (or 'baseline' if no balancing).")
    metric_name_raw: str = Field(description="Metric name verbatim (e.g., 'macro F1', 'Balanced Accuracy').")
    metric_value: float = Field(description="Numerical metric value. Convert percentages to decimals (e.g., 84.5% -> 0.845).")
    is_baseline_raw: bool = Field(description="True if paper explicitly marks this as a no-balancing baseline.")

    # Caracterização do dataset (opcional)
    dataset_size_raw: Optional[str] = Field(default=None, description="Number of instances as written (e.g., '70,000 samples'). Null if not stated.")
    dataset_num_classes_raw: Optional[str] = Field(default=None, description="Number of classes as written. Null if not stated.")
    dataset_imbalance_ratio_raw: Optional[str] = Field(default=None, description="Imbalance ratio or class distribution as written (e.g., 'IR=100', 'minority 5%'). Null if not stated.")
    dataset_domain_raw: Optional[str] = Field(default=None, description="Domain (e.g., 'medical imaging', 'fraud detection'). Null if not clear.")
    task_type_raw: Optional[str] = Field(default=None, description="Task type (e.g., 'binary classification'). Null if not specified.")

    # Modelo (opcional)
    model_hparams_raw: Optional[str] = Field(default=None, description="Hyperparameters/training setup if mentioned. Null otherwise.")

    # Métrica (opcional)
    metric_split_raw: Optional[str] = Field(default=None, description="'test', 'val', 'cv' or other split as written. Null if unclear.")
    metric_aggregation_raw: Optional[str] = Field(default=None, description="How the value was aggregated (e.g., 'mean', 'mean±std', 'best'). Null if unclear.")

    # Auditoria (obrigatórios)
    extracted_evidence: str = Field(description="Short verbatim quote (1-2 sentences) from the paper supporting this extraction.")
    extraction_confidence: float = Field(description="Your confidence 0.0-1.0 that this is a correct, complete extraction.")
    notes: Optional[str] = Field(default=None, description="Anything notable about this configuration. Null otherwise.")


class PaperExtraction(BaseModel):
    configurations: List[Configuration] = Field(description="All experimental configurations extracted from the paper. Empty list if paper has no usable data.")
    paper_notes: Optional[str] = Field(default=None, description="General notes about the paper, especially if the configurations list is empty.")

In [ ]:
SYSTEM_PROMPT = """You are extracting structured experimental data from machine learning papers that study class-balancing strategies for supervised classification.

For each paper, extract every experimental CONFIGURATION reported: a unique combination of (dataset, model, balancing strategy) measured by at least one metric value.

A single paper typically has multiple configurations. If a paper reports a results table like:
  Dataset A: ResNet + SMOTE=0.82, ResNet + Focal Loss=0.85, ResNet baseline=0.71
  Dataset B: ResNet + SMOTE=0.74, ResNet + Focal Loss=0.78, ResNet baseline=0.65
Then there are 6 configurations to extract.

REQUIRED for every configuration:
- dataset_name_raw: dataset name verbatim
- model_name_raw: model architecture verbatim
- balancing_strategy_raw: strategy name verbatim (or "baseline" / "no balancing" if that's the configuration)
- metric_name_raw: metric name verbatim (e.g., "macro F1", "Balanced Accuracy", "TPR gap", "AUPRC")
- metric_value: numerical value as a float. Convert percentages to decimals: "84.5%" -> 0.845
- is_baseline_raw: TRUE only if the paper EXPLICITLY marks this as a no-balancing baseline (looks for "baseline", "vanilla", "no resampling", "unmodified", "without balancing"). Otherwise FALSE.
- extracted_evidence: a short verbatim quote (1-2 sentences max) from the paper supporting this extraction
- extraction_confidence: your confidence 0.0-1.0

OPTIONAL (use null if not clearly stated):
- dataset_size_raw, dataset_num_classes_raw, dataset_imbalance_ratio_raw, dataset_domain_raw, task_type_raw
- model_hparams_raw
- metric_split_raw, metric_aggregation_raw
- notes

IMPORTANT RULES:
1. SKIP configurations missing any of: dataset, model, strategy, metric value. Don't make up values.
2. Multiple metrics for the same (dataset, model, strategy) = MULTIPLE configurations (one per metric).
3. Extract EVERYTHING reported in the paper's results tables — do NOT filter to "important" ones. A paper with 60 rows in a results table should yield 60 configurations.
4. If the paper turns out to have no usable experimental data on class-balancing (e.g., it's a theory paper, or balancing is only mentioned in related work), return configurations=[] and explain in paper_notes.
5. The paper text may be truncated in the middle — work with what's available."""

print(f"System prompt: {len(SYSTEM_PROMPT)} chars (~{len(SYSTEM_PROMPT)//4} tokens)")

In [ ]:
def extract_configurations_from_paper(paper_id: str, pdf_path: str,
                                       model: str = MODEL,
                                       max_retries: int = MAX_RETRIES) -> dict:
    """Extrai texto do PDF e roda LLM com structured output. Retorna sempre um dict."""
    base = {
        "paper_id": paper_id,
        "configurations": None,
        "paper_notes": None,
        "input_tokens": None,
        "output_tokens": None,
        "stop_reason": None,
        "n_configs": 0,
        "error": None,
    }

    # 1) Extrair e truncar texto
    try:
        raw_text = extract_pdf_text(pdf_path)
        text = smart_truncate(raw_text)
    except Exception as e:
        base["error"] = f"pdf_extract_failed: {type(e).__name__}: {str(e)[:200]}"
        return base

    if len(text) < 500:
        base["error"] = f"text_too_short ({len(text)} chars after extraction)"
        return base

    # 2) Chamar LLM com rate limit
    user_msg = f"Paper text follows. Extract all experimental configurations.\n\n---\n\n{text}"
    rate_limiter.acquire()
    try:
        response = client.with_options(max_retries=max_retries).messages.parse(
            model=model,
            max_tokens=MAX_TOKENS_OUTPUT,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
            output_format=PaperExtraction,
        )
        base["input_tokens"] = response.usage.input_tokens
        base["output_tokens"] = response.usage.output_tokens
        base["stop_reason"] = response.stop_reason

        parsed = response.parsed_output
        if parsed is None:
            base["error"] = f"no_parsed_output (stop_reason={response.stop_reason})"
            return base

        base["configurations"] = [c.model_dump() for c in parsed.configurations]
        base["paper_notes"] = parsed.paper_notes
        base["n_configs"] = len(parsed.configurations)
        return base
    except Exception as e:
        base["error"] = f"{type(e).__name__}: {str(e)[:300]}"
        return base

## Smoke test — 3 papers

Antes de torrar ~$23 nos 600, verifique que a extração tá sensata: olhe quantas configurações foram extraídas, leia 1-2 `extracted_evidence` pra confirmar que estão sendo capturadas as métricas reais do paper, e cheque `extraction_confidence`.

In [ ]:
sample = df_ext.sample(n=3, random_state=42).reset_index(drop=True)

for row in sample.itertuples():
    print("=" * 80)
    print(f"[{row.rank}] {row.title[:90]}")
    print(f"  PDF: {row.pdf_local_path}")
    result = extract_configurations_from_paper(row.paper_id, row.pdf_local_path)
    if result["error"]:
        print(f"  ERRO: {result['error']}")
        continue
    print(f"  configs extraídas: {result['n_configs']}")
    print(f"  tokens: in={result['input_tokens']} out={result['output_tokens']}")
    if result["paper_notes"]:
        print(f"  paper_notes: {result['paper_notes'][:200]}")
    if result["n_configs"] > 0:
        print(f"  --- Primeira config: ---")
        first = result["configurations"][0]
        for k in ["dataset_name_raw", "model_name_raw", "balancing_strategy_raw",
                  "metric_name_raw", "metric_value", "is_baseline_raw",
                  "extracted_evidence", "extraction_confidence"]:
            print(f"    {k}: {first.get(k)}")
    print()

## Run completo com checkpoint

- `load_done_ids` só considera entradas com `error is None` — erros são re-tentados.
- Cada paper grava no JSONL imediatamente após processar (append + flush).
- A SDK do Anthropic auto-retentativa em 429/5xx; o `RateLimiter` evita o 429 em primeiro lugar.

In [ ]:
def load_done_ids(ckpt_path: Path) -> set:
    if not ckpt_path.exists():
        return set()
    done = set()
    with open(ckpt_path, encoding="utf-8") as f:
        for line in f:
            try:
                entry = json.loads(line)
                if entry.get("error") is None and entry.get("configurations") is not None:
                    done.add(entry["paper_id"])
            except Exception:
                pass
    return done


def run_extraction(df: pd.DataFrame, ckpt_path: Path,
                    n_workers: int = N_WORKERS, model: str = MODEL):
    done_ids = load_done_ids(ckpt_path)
    todo = df[~df["paper_id"].isin(done_ids)].copy()
    print(f"Total: {len(df):>5}")
    print(f"Sucessos prévios: {len(done_ids):>5}")
    print(f"Restantes (inclui erros prévios): {len(todo):>5}")

    if len(todo) == 0:
        return 0

    est_tokens_in = len(todo) * 7500
    est_tokens_out = len(todo) * 1000
    est_cost = est_tokens_in / 1_000_000 * 3.0 + est_tokens_out / 1_000_000 * 15.0
    est_time = len(todo) * 60 / RPM_LIMIT
    print(f"\nCusto estimado dos restantes: ~${est_cost:.2f}")
    print(f"Tempo mínimo (limitado por {RPM_LIMIT} RPM): ~{est_time/60:.1f} min\n")

    start = time.time()
    n_success = 0
    n_error = 0
    total_configs = 0

    with open(ckpt_path, "a", encoding="utf-8") as f_out:
        with ThreadPoolExecutor(max_workers=n_workers) as pool:
            futures = {
                pool.submit(extract_configurations_from_paper,
                            row.paper_id, row.pdf_local_path, model): row.paper_id
                for row in todo.itertuples()
            }
            for fut in tqdm(as_completed(futures), total=len(futures), desc="Extraindo"):
                try:
                    result = fut.result()
                except Exception as e:
                    result = {
                        "paper_id": futures[fut],
                        "configurations": None, "paper_notes": None,
                        "input_tokens": None, "output_tokens": None,
                        "stop_reason": None, "n_configs": 0,
                        "error": f"future_exception: {type(e).__name__}: {e}",
                    }
                f_out.write(json.dumps(result, ensure_ascii=False) + "\n")
                f_out.flush()
                if result.get("error") is None:
                    n_success += 1
                    total_configs += result.get("n_configs", 0)
                else:
                    n_error += 1

    elapsed = time.time() - start
    print(f"\nProcessados: {n_success + n_error}")
    print(f"  Sucesso:    {n_success}")
    print(f"  Erro:       {n_error}")
    print(f"  Configurações totais extraídas: {total_configs}")
    print(f"  Tempo: {elapsed:.1f}s ({(n_success+n_error)/max(elapsed,1):.2f} req/s)")
    return n_success + n_error

In [ ]:
n_processed = run_extraction(df_ext, CKPT_PATH)

## Achatar checkpoints em `configurations_raw.parquet`

O JSONL tem uma linha por paper (com uma lista de configurações dentro). Para análise causal, queremos uma linha por configuração. Achatamos:

- Para cada paper com sucesso, expandimos cada configuração em uma linha.
- Adicionamos `paper_id` e gerar `config_id` único (`{paper_id}_{idx}`).
- Salvamos como parquet.

In [ ]:
def load_checkpoints_extraction(ckpt_path: Path) -> pd.DataFrame:
    rows = []
    if not ckpt_path.exists():
        return pd.DataFrame(rows)
    with open(ckpt_path, encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return pd.DataFrame(rows)


df_ckpt = load_checkpoints_extraction(CKPT_PATH)

# Dedup por paper_id, preferindo sucesso
df_ckpt["_is_success"] = df_ckpt["error"].isna() & df_ckpt["configurations"].notna()
df_ckpt["_order"] = range(len(df_ckpt))
df_ckpt = df_ckpt.sort_values(["paper_id", "_is_success", "_order"],
                               ascending=[True, True, True])
df_ckpt = df_ckpt.drop_duplicates("paper_id", keep="last").reset_index(drop=True)
df_ckpt = df_ckpt.drop(columns=["_is_success", "_order"])

print(f"Papers no checkpoint: {len(df_ckpt)}")
print(f"Com configurações: {df_ckpt['n_configs'].fillna(0).gt(0).sum()}")
print(f"Vazios (sem configurações utilizáveis): {df_ckpt['n_configs'].fillna(0).eq(0).sum() - df_ckpt['error'].notna().sum()}")
print(f"Com erro: {df_ckpt['error'].notna().sum()}")
print(f"Total de configurações extraídas: {int(df_ckpt['n_configs'].fillna(0).sum())}")

# Achatar
config_rows = []
for row in df_ckpt.itertuples():
    if not row.configurations:
        continue
    for idx, cfg in enumerate(row.configurations):
        flat = {"config_id": f"{row.paper_id}#{idx}", "paper_id": row.paper_id}
        flat.update(cfg)
        config_rows.append(flat)

df_configs = pd.DataFrame(config_rows)
df_configs.to_parquet(CONFIG_PATH, index=False)
print(f"\nSalvo em {CONFIG_PATH}")
print(f"Linhas (configurações): {len(df_configs)}")

## Resumo

Quero ver:
- Distribuição de #configs por paper — esperamos uma cauda longa (alguns papers com 50+, outros com 2-3).
- Top datasets, modelos, estratégias mais frequentes (já em forma raw — vai dar pra perceber a normalização que vamos precisar).
- Distribuição de `extraction_confidence`.
- Quantos papers retornaram lista vazia + sample de `paper_notes`.

In [ ]:
# Distribuição de #configs por paper
configs_per_paper = df_configs.groupby("paper_id").size()
print(f"Papers com >=1 config: {len(configs_per_paper)}")
print(f"Mediana de configs/paper: {configs_per_paper.median()}")
print(f"Média:                   {configs_per_paper.mean():.1f}")
print(f"Máx:                     {configs_per_paper.max()}")
print(f"\nDistribuição de configs/paper:")
print(configs_per_paper.describe([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

print(f"\nTop 10 datasets (raw — pré-normalização):")
print(df_configs["dataset_name_raw"].value_counts().head(10))

print(f"\nTop 10 modelos (raw):")
print(df_configs["model_name_raw"].value_counts().head(10))

print(f"\nTop 10 estratégias (raw):")
print(df_configs["balancing_strategy_raw"].value_counts().head(10))

print(f"\nTop 10 métricas (raw):")
print(df_configs["metric_name_raw"].value_counts().head(10))

print(f"\nDistribuição de extraction_confidence:")
print(df_configs["extraction_confidence"].describe())

print(f"\n% configurações marcadas como baseline:")
print(df_configs["is_baseline_raw"].value_counts(normalize=True))

In [ ]:
# Papers sem configurações utilizáveis — útil pra auditar
empty_papers = df_ckpt[df_ckpt["n_configs"].fillna(0).eq(0) & df_ckpt["error"].isna()]
if len(empty_papers) > 0:
    print(f"\nPapers que retornaram lista vazia ({len(empty_papers)}):")
    print("Sample de paper_notes (até 5):")
    for row in empty_papers.head(5).itertuples():
        print(f"  - {row.paper_id}: {(row.paper_notes or '<sem nota>')[:200]}")

# Top categorias de erro
errors = df_ckpt[df_ckpt["error"].notna()]
if len(errors) > 0:
    print(f"\nTop categorias de erro ({len(errors)} total):")
    err_cat = errors["error"].astype(str).str.split(":").str[0]
    print(err_cat.value_counts().head(10))

# Custo total real
total_in = int(df_ckpt["input_tokens"].fillna(0).sum())
total_out = int(df_ckpt["output_tokens"].fillna(0).sum())
cost = total_in / 1_000_000 * 3.0 + total_out / 1_000_000 * 15.0
print(f"\nTokens totais — input: {total_in:>9,}  output: {total_out:>9,}")
print(f"Custo total estimado: ~${cost:.2f}")

## Próximo passo

`05_normalizacao.ipynb` — pega `configurations_raw.parquet` (campos `*_raw` cheios de variações textuais como "SMOTE", "smote", "Smote with k=5", "Synthetic Minority Over-sampling Technique") e normaliza para o schema canônico de `experiments` (definido em `00_design.ipynb`):

- `dataset_canonical`: vocabulário controlado (CIFAR-10-LT, ImageNet-LT, Adult, etc.) — combinação de lista curada + matching LLM nos casos ambíguos.
- `model_family`: enum (`cnn`, `transformer`, `gbm`, `linear`, ...).
- `balancing_strategy`: enum (`none`, `oversampling`, `undersampling`, `cost_sensitive`, `hybrid`, ...).
- `metric_f1_macro`, `metric_balanced_acc`, `metric_tpr_gap`, etc.: pivota `metric_name_raw` + `metric_value` em colunas tipadas.
- Aplicar critérios de aceitação (mínimo de campos preenchidos, métrica utilizável).
- Construir `within_paper_group_id` para identificar configurações pareadas (mesmo paper, dataset, modelo).

Me passa o resumo desta etapa (n de configs extraídas, top datasets/modelos/estratégias raw, custo real) e a gente segue.